# Milestone 2

This milestone focuses on advanced audio analysis and feature extraction using **librosa** and audio processing techniques.

---

## Suggested Readings
- [Librosa Advanced Features](https://librosa.org/doc/main/feature.html)
- [Audio Feature Extraction](https://musicinformationretrieval.com/feature_extraction.html)

---

## Instructions
Use this notebook to answer **all Milestone-2 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch
import soundfile as sf

import warnings
warnings.filterwarnings("ignore")

In [ ]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [ ]:
# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
NOISE_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/noise'
MASHUP_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups'
GENRES = ['blues','classical','country','disco','hiphop','jazz','metal','pop','reggae','rock']
STEMS = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']

In [ ]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    rng = random.Random(seed)
    
    all_songs = []
    
    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.exists(genre_path):
            continue
            
        song_folders = [f for f in os.listdir(genre_path) if os.path.isdir(os.path.join(genre_path, f))]
        song_folders.sort()
        
        for song_folder in song_folders:
            song_path = os.path.join(genre_path, song_folder)
            song_files = os.listdir(song_path)
            missing_stems = []
            
            for stem in STEMS:
                if stem not in song_files:
                    missing_stems.append(stem)
            
            if not missing_stems:
                song_data = {
                    'genre': genre,
                    'song_folder': song_folder,
                    'stems': {stem.replace('.wav', ''): os.path.join(song_path, stem) for stem in STEMS}
                }
                all_songs.append(song_data)
    
    for genre in GENRES:
        genre_songs = [s for s in all_songs if s['genre'] == genre]
        rng.shuffle(genre_songs)
        split_idx = int(len(genre_songs) * (1 - val_split))
        train_songs = genre_songs[:split_idx]
        val_songs = genre_songs[split_idx:]
        
        def add_to_dict(target_dict, song_list):
            for song in song_list:
                for stem_name, stem_path in song['stems'].items():
                    target_dict[song['genre']][stem_name].append(stem_path)
        
        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)
    
    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)
print(f"Training dataset built with {len(tr)} genres")
print(f"Validation dataset built with {len(val)} genres")

In [ ]:
def analyze_jazz_duration(train_dataset):
    """Calculate mean duration of Jazz genre stems in train dataset"""
    jazz_durations = []
    
    for stem_name in train_dataset['jazz']:
        for file_path in train_dataset['jazz'][stem_name]:
            try:
                # Get duration without loading the full audio
                duration = librosa.get_duration(filename=file_path)
                jazz_durations.append(duration)
            except Exception:
                continue
    
    mean_duration = np.mean(jazz_durations)
    print(f"Q1: Mean duration of Jazz stems: {mean_duration:.2f} seconds")
    return mean_duration

jazz_mean_duration = analyze_jazz_duration(tr)

In [ ]:
def get_unique_sample_rates():
    """Get unique sample rates from all datasets"""
    sample_rates = set()
    
    # Check genres_stems
    for genre in GENRES:
        genre_path = os.path.join(DATA_ROOT, genre)
        if os.path.exists(genre_path):
            for song_folder in os.listdir(genre_path):
                song_path = os.path.join(genre_path, song_folder)
                if os.path.isdir(song_path):
                    for stem_file in os.listdir(song_path):
                        if stem_file.endswith('.wav'):
                            try:
                                file_path = os.path.join(song_path, stem_file)
                                sr = librosa.get_samplerate(file_path)
                                sample_rates.add(sr)
                            except Exception:
                                continue
    
    # Check noise data
    if os.path.exists(NOISE_ROOT):
        for noise_file in os.listdir(NOISE_ROOT):
            if noise_file.endswith('.wav'):
                try:
                    file_path = os.path.join(NOISE_ROOT, noise_file)
                    sr = librosa.get_samplerate(file_path)
                    sample_rates.add(sr)
                except Exception:
                    continue
    
    # Check mashups
    if os.path.exists(MASHUP_ROOT):
        for mashup_file in os.listdir(MASHUP_ROOT):
            if mashup_file.endswith('.wav'):
                try:
                    file_path = os.path.join(MASHUP_ROOT, mashup_file)
                    sr = librosa.get_samplerate(file_path)
                    sample_rates.add(sr)
                except Exception:
                    continue
    
    sorted_rates = sorted(list(sample_rates))
    print(f"Q2: Unique sample rates: {sorted_rates}")
    return sorted_rates

unique_sr = get_unique_sample_rates()

In [ ]:
def count_empty_files(train_dataset):
    """Count empty or zero-byte audio files in train dataset"""
    empty_count = 0
    total_files = 0
    
    for genre in train_dataset:
        for stem_name in train_dataset[genre]:
            for file_path in train_dataset[genre][stem_name]:
                total_files += 1
                try:
                    file_size = os.path.getsize(file_path)
                    if file_size == 0:
                        empty_count += 1
                except Exception:
                    continue
    
    print(f"Q3: Empty/zero-byte files: {empty_count} out of {total_files} total files")
    return empty_count

empty_files = count_empty_files(tr)

In [ ]:
def calculate_vocal_peak_amplitude(train_dataset):
    """Calculate average peak amplitude in dB for vocal stems"""
    peak_amplitudes = []
    
    for genre in train_dataset:
        for file_path in train_dataset[genre]['vocals']:
            try:
                audio, sr_loaded = librosa.load(file_path, sr=None)  # Keep original SR
                # Calculate peak amplitude in dB
                peak_amp = np.max(np.abs(audio))
                if peak_amp > 0:
                    peak_db = 20 * np.log10(peak_amp)
                    peak_amplitudes.append(peak_db)
            except Exception:
                continue
    
    avg_peak_db = np.mean(peak_amplitudes)
    print(f"Q4: Average peak amplitude for vocals: {avg_peak_db:.2f} dB")
    return avg_peak_db

vocal_peak_amp = calculate_vocal_peak_amplitude(tr)

In [ ]:
def calculate_spectral_centroid(train_dataset):
    """Calculate spectral centroid for blues genre and find genre with highest mean"""
    genre_centroids = {}
    blues_centroids = []
    
    for genre in train_dataset:
        all_centroids = []
        
        for stem_name in train_dataset[genre]:
            for file_path in train_dataset[genre][stem_name]:
                try:
                    audio, sr_loaded = librosa.load(file_path, sr=SR)
                    # Calculate spectral centroid
                    cent = librosa.feature.spectral_centroid(y=audio, sr=sr_loaded)
                    mean_centroid = np.mean(cent)
                    all_centroids.append(mean_centroid)
                    
                    if genre == 'blues':
                        blues_centroids.append(mean_centroid)
                except Exception:
                    continue
        
        if all_centroids:
            genre_centroids[genre] = np.mean(all_centroids)
    
    # Q5: Blues mean spectral centroid
    blues_mean = np.mean(blues_centroids) if blues_centroids else 0
    print(f"Q5: Mean spectral centroid for blues: {blues_mean:.2f}")
    
    # Q6: Genre with highest mean spectral centroid
    if genre_centroids:
        highest_genre = max(genre_centroids, key=genre_centroids.get)
        highest_value = genre_centroids[highest_genre]
        print(f"Q6: Genre with highest spectral centroid: {highest_genre} ({highest_value:.2f})")
    
    return blues_mean, genre_centroids

blues_centroid, all_centroids = calculate_spectral_centroid(tr)

In [ ]:
def count_initial_silence(train_dataset, threshold=1e-4, time_window=0.5):
    """Count files with silence in first 0.5 seconds"""
    silent_files = 0
    total_files = 0
    samples_to_check = int(SR * time_window)
    
    for genre in train_dataset:
        for stem_name in train_dataset[genre]:
            for file_path in train_dataset[genre][stem_name]:
                total_files += 1
                try:
                    audio, sr_loaded = librosa.load(file_path, sr=SR)
                    # Check first 0.5 seconds
                    if len(audio) >= samples_to_check:
                        initial_segment = audio[:samples_to_check]
                        max_amplitude = np.max(np.abs(initial_segment))
                        if max_amplitude < threshold:
                            silent_files += 1
                except Exception:
                    continue
    
    print(f"Q7: Files with silence in first 0.5s: {silent_files} out of {total_files}")
    return silent_files

initial_silence_count = count_initial_silence(tr)

In [ ]:
# Additional imports for Part 2
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import seaborn as sns

In [ ]:
# --- 1. Setup and Preprocessing for Part 2 ---
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES_ML = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

def extract_features(song_path):
    # Load 10s at 22050Hz
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]

In [ ]:
# --- 2. Data Preparation & Stratified Split ---
data = []
for g in GENRES_ML:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    for s in songs[:50]: # Sampling 50 for speed; use all for final
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Genres: {df['genre'].nunique()}")

In [ ]:
# --- 3. Model Training (Decision Tree) ---
print("Extracting features for training data...")
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']

print("Extracting features for validation data...")
X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']

print("Training Decision Tree classifier...")
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

print("Model training completed!")

In [ ]:
# --- 4. Model Evaluation ---
# Compute predictions and metrics
y_pred = clf.predict(X_val)
macro_f1 = f1_score(y_val, y_pred, average='macro')
cm = confusion_matrix(y_val, y_pred)
cr = classification_report(y_val, y_pred)

print(f"Q8: Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Detailed Classification Report:")
print(cr)

# Calculate accuracy
accuracy = (y_pred == y_val).mean()
print(f"Q11: Model Accuracy: {accuracy:.4f}")

In [ ]:
# --- 5. Confusion Matrix Analysis ---
# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=GENRES_ML, yticklabels=GENRES_ML)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Calculate TP, TN, FP, FN for each genre
tp_counts = {}
fn_counts = {}

for i, genre in enumerate(GENRES_ML):
    tp_counts[genre] = cm[i, i]  # True Positives
    fn_counts[genre] = cm[i, :].sum() - cm[i, i]  # False Negatives

# Find answers for specific questions
highest_tp_genre = max(tp_counts, key=tp_counts.get)
lowest_fn_genre = min(fn_counts, key=fn_counts.get)

print(f"Q12: Genre with highest true positives: {highest_tp_genre} ({tp_counts[highest_tp_genre]})")
print(f"Q13: Genre with lowest false negatives: {lowest_fn_genre} ({fn_counts[lowest_fn_genre]})")

# Extract specific values from classification report
report_dict = classification_report(y_val, y_pred, output_dict=True)
hiphop_precision = report_dict['hiphop']['precision']
pop_recall = report_dict['pop']['recall']

print(f"Q9: Precision of hiphop: {hiphop_precision:.4f}")
print(f"Q10: Recall of pop: {pop_recall:.4f}")

In [ ]:
# Summary of all Part 2 answers
print("\n" + "="*50)
print("MILESTONE 2 PART 2 ANSWERS SUMMARY")
print("="*50)
print(f"Q8: Validation Macro F1 Score: {macro_f1:.4f}")
print(f"Q9: Precision of hiphop: {hiphop_precision:.4f}")
print(f"Q10: Recall of pop: {pop_recall:.4f}")
print(f"Q11: Model Accuracy: {accuracy:.4f}")
print(f"Q12: Genre with highest true positives: {highest_tp_genre}")
print(f"Q13: Genre with lowest false negatives: {lowest_fn_genre}")

# Detailed TP/FN analysis
print("\nDetailed True Positives by Genre:")
for genre in GENRES_ML:
    print(f"  {genre}: {tp_counts[genre]}")

print("\nDetailed False Negatives by Genre:")
for genre in GENRES_ML:
    print(f"  {genre}: {fn_counts[genre]}")

In [ ]:
# Milestone 2 - Part 2: Machine Learning

This section focuses on building a Decision Tree classifier for genre classification using audio features.

---

## Instructions
Complete the code and answer the machine learning questions below.